# AAPL historical news — comprehensive extraction, audit & FinBERT

**Purpose:** Rebuild `AAPL_news.csv` by scanning the **full** FNSPID news dataset rather than the truncated/community Nasdaq-100 sample that gave missing years. Extract **AAPL only**, check calendar-year coverage, deduplicate, run FinBERT, and export data for `AAPL_Stock_Forecasting_Fixed_News_Ready.ipynb`.

**Realistic caveats:** This is a **large public dataset (~10.6 GB of Parquet files; over 15 million source rows)** and scanning it via Colab streaming can be slow. The notebook does **not promise** full coverage: it reports real counts first and stops if coverage fails. FNSPID dates may be date-only or wrong. A date delay is not sufficient to prove an article was available historically; audit any suspicious records and treat forecasting results as exploratory.

Source: [FNSPID Nasdaq full mirror](https://huggingface.co/datasets/sabareesh88/FNSPID_nasdaq); [original FNSPID project](https://huggingface.co/datasets/Zihan1004/FNSPID). Dataset is CC BY-NC 4.0: **non-commercial use only**.

Run in **Colab** (GPU accelerator recommended for FinBERT). Files generated: `AAPL_full_news_raw.csv`, `AAPL_news_audit.csv`, `AAPL_news.csv`, `AAPL_news_year_counts.csv`.

In [1]:
%pip uninstall -y torchvision
%pip install -q "transformers==4.44.2" "huggingface_hub==0.24.7" "tokenizers==0.19.1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Everything imported successfully!")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Everything imported successfully!


In [2]:
from pathlib import Path
import pandas as pd, numpy as np, re, os, json, warnings
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset
OUT=Path('/content') if Path('/content').exists() else Path.cwd()
OUT.mkdir(exist_ok=True,parents=True)
TICKER='AAPL'
START_YEAR, END_YEAR = 2013, 2022
DATASET_ID='sabareesh88/FNSPID_nasdaq'
RAW_PATH=OUT/'AAPL_full_news_raw.csv'
print('Output:', OUT)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Output: /content


## 1. Extract AAPL headlines by streaming the full public mirror
This reads the dataset sequentially and retains only AAPL records. It does **not** load the entire news corpus into memory. If Colab disconnects, keep `AAPL_full_news_raw.csv` in Drive after a successful run; subsequent runs reuse it. Data source field names are normalized defensively.

In [3]:
if RAW_PATH.exists():
    raw=pd.read_csv(RAW_PATH, dtype=str)
    print('Using cached file:', RAW_PATH, 'rows:', len(raw))
else:
    stream=load_dataset(DATASET_ID, split='train', streaming=True)
    records=[]
    scanned=0
    for item in tqdm(stream, desc='Scanning full financial-news corpus'):
        scanned+=1
        ticker=str(item.get('Stock_symbol', item.get('stock_symbol',item.get('ticker','')))).strip().upper()
        if ticker!=TICKER: continue
        records.append({
            'source_timestamp':item.get('Date',item.get('date')),
            'headline':item.get('Article_title',item.get('title','')),
            'url':item.get('Url',item.get('url','')),
            'publisher':item.get('Publisher',item.get('publisher','')),
            'ticker':ticker
        })
    raw=pd.DataFrame(records)
    if raw.empty: raise RuntimeError('No AAPL records found. Verify dataset schema and source.')
    raw.to_csv(RAW_PATH,index=False)
    print('Scanned records:', scanned)
print('AAPL records before cleaning:', len(raw))
print(raw.head(3).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/965 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Scanning full financial-news corpus: 0it [00:00, ?it/s]

Scanned records: 15549299
AAPL records before cleaning: 9338
       source_timestamp                                                                                                        headline                                                                                                                    url publisher ticker
2023-12-16 22:00:00 UTC My 6 Largest Portfolio Holdings Heading Into 2024 -- and the Important Investing Lesson I Learned From Each One https://www.nasdaq.com/articles/my-6-largest-portfolio-holdings-heading-into-2024-and-the-important-investing-lesson-i             AAPL
2023-12-16 22:00:00 UTC                                       Brokers Suggest Investing in Apple (AAPL): Read This Before Placing a Bet              https://www.nasdaq.com/articles/brokers-suggest-investing-in-apple-aapl%3A-read-this-before-placing-a-bet             AAPL
2023-12-16 21:00:00 UTC                                                                                   Company News for 

## 2. Clean and audit source timestamps
Important: Keep the **original timestamp**, headline, URL and year counts in the audit output. Publication date is not necessarily publication **time**. Date-only news is usable only with delayed alignment. Future event dates mentioned in titles are not always evidence of leakage (e.g., previews); this check flags records for review rather than claiming they are invalid.

In [4]:
news=raw.copy()
news['source_timestamp']=news['source_timestamp'].astype(str)
news['published_at']=pd.to_datetime(news['source_timestamp'],utc=True,errors='coerce',format='mixed')
news['headline']=news['headline'].fillna('').astype(str).str.strip()
news['url']=news['url'].fillna('').astype(str).str.strip()
news=news.dropna(subset=['published_at'])
news=news[news['headline'].str.len().ge(12)].copy()
news=news[news['published_at'].dt.year.between(START_YEAR,END_YEAR)].copy()
news['clean_headline']=news['headline'].str.lower().str.replace(r'\s+',' ',regex=True)
news=news.drop_duplicates(subset=['published_at','clean_headline'])
news['source_year']=news['published_at'].dt.year
news['date_only_or_midnight']=news['published_at'].dt.hour.eq(0)&news['published_at'].dt.minute.eq(0)&news['published_at'].dt.second.eq(0)
# A simple, conservative textual contradiction flag, NOT a verified publication time
explicit_years=news['headline'].str.findall(r'\b(?:20[0-3][0-9])\b')
news['mentions_later_year']= [any(int(y)>int(pub.year) for y in matches) for matches,pub in zip(explicit_years,news['published_at'])]
news['timestamp_caution']=news['mentions_later_year']
counts=news.groupby('source_year').agg(articles=('headline','size'),unique_dates=('published_at',lambda s:s.dt.date.nunique()),midnight_fraction=('date_only_or_midnight','mean'),flagged=('timestamp_caution','sum')).reset_index()
counts.to_csv(OUT/'AAPL_news_year_counts.csv',index=False)
print(counts.to_string(index=False))
print('Missing years:',sorted(set(range(START_YEAR,END_YEAR+1))-set(counts.source_year)))
print('Suspicious later-year title examples:')
print(news.loc[news['timestamp_caution'],['published_at','headline']].head(12).to_string(index=False))
news.drop(columns=['clean_headline']).to_csv(OUT/'AAPL_full_news_clean_audit.csv',index=False)

 source_year  articles  unique_dates  midnight_fraction  flagged
        2020       471            81           0.978769        6
        2022      3018           208           1.000000       83
Missing years: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021]
Suspicious later-year title examples:
             published_at                                                                      headline
2022-12-31 00:00:00+00:00                             4 Red-Hot Growth Stocks to Buy in 2023 and Beyond
2022-12-30 00:00:00+00:00           Have $500? These 2 Stocks Could Be Bargain Buys for 2023 and Beyond
2022-12-30 00:00:00+00:00       Earnings Outlook for 2023 and Featured Reports for Apple, JNJ, & Others
2022-12-30 00:00:00+00:00                           3 Top Stocks Worth Buying for the Long Term in 2023
2022-12-29 00:00:00+00:00                                      AAPL February 2023 Options Begin Trading
2022-12-29 00:00:00+00:00                                  Will Apple (AAPL) St

## 3. Choose whether the coverage meets the research goal
For a 2013–2022 full-window experiment, sparse calendar years are problematic. This cell clearly identifies them. If coverage is missing, **do not** interpret missing sentiment as neutral headlines. You can still run a proof of concept, but the full-period comparison will be weak.

In [5]:
MIN_ARTICLES_PER_YEAR=100  # adjustable research-quality screening, not a source guarantee
by_year=dict(zip(counts.source_year,counts.articles))
sparse={y:int(by_year.get(y,0)) for y in range(START_YEAR,END_YEAR+1) if by_year.get(y,0)<MIN_ARTICLES_PER_YEAR}
print('Years below threshold:',sparse or 'none')
print('Total deduplicated records:',len(news))
print('Date-only/midnight share:',round(news.date_only_or_midnight.mean()*100,2),'%')
if sparse:
    warnings.warn('Coverage is incomplete/sparse. Do not claim a full-coverage 2013–2022 news study.')

Years below threshold: {2013: 0, 2014: 0, 2015: 0, 2016: 0, 2017: 0, 2018: 0, 2019: 0, 2021: 0}
Total deduplicated records: 3489
Date-only/midnight share: 99.71 %


/tmp/ipykernel_1136/2178789090.py:8: UserWarning: Coverage is incomplete/sparse. Do not claim a full-coverage 2013–2022 news study.
  warnings.warn('Coverage is incomplete/sparse. Do not claim a full-coverage 2013–2022 news study.')


In [10]:
from google.colab import files

if "news" in globals():
    news.to_csv("/content/AAPL_news_backup.csv", index=False)
    files.download("/content/AAPL_news_backup.csv")
elif "raw" in globals():
    raw.to_csv("/content/AAPL_news_backup.csv", index=False)
    files.download("/content/AAPL_news_backup.csv")
else:
    print("No news DataFrame found. Check your saved CSV files.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. FinBERT: headlines → financial sentiment
Scores are `P(positive) − P(negative)` in [−1,+1]. Pretrained FinBERT is applied to headlines without fitting to your test-price data. **This is language sentiment, not a verified stock-return signal**.

In [3]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

news = pd.read_csv("AAPL_news_backup.csv")

print("Articles loaded:", len(news))
print(news.columns.tolist())
print(news.head())

Saving AAPL_news_backup.csv to AAPL_news_backup (1).csv
Articles loaded: 3489
['source_timestamp', 'headline', 'url', 'publisher', 'ticker', 'published_at', 'clean_headline', 'source_year', 'date_only_or_midnight', 'mentions_later_year', 'timestamp_caution']
          source_timestamp                                           headline  \
0  2022-12-31 00:00:00 UTC               Where Will Apple Stock Be in 1 Year?   
1  2022-12-31 00:00:00 UTC     Where Will Unity Software Stock Be in 3 Years?   
2  2022-12-31 00:00:00 UTC  4 Red-Hot Growth Stocks to Buy in 2023 and Beyond   
3  2022-12-31 00:00:00 UTC  Even in an Advertising Slowdown, These 3 Stock...   
4  2022-12-31 00:00:00 UTC  3 Unstoppable Growth Stocks to Buy After a Sto...   

                                                 url publisher ticker  \
0  https://www.nasdaq.com/articles/where-will-app...       NaN   AAPL   
1  https://www.nasdaq.com/articles/where-will-uni...       NaN   AAPL   
2  https://www.nasdaq.com/articles/

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

# Check news data
if news.empty:
    raise RuntimeError("No articles to score")

# Select GPU or CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Load FinBERT
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

# Prepare headlines
texts = news["headline"].fillna("").astype(str).tolist()

BATCH = 32 if device.type == "cuda" else 8
values = []

# Calculate sentiment
for i in tqdm(
    range(0, len(texts), BATCH),
    desc="Scoring AAPL headlines"
):

    batch = texts[i:i+BATCH]

    inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

    for probs in probabilities:
        scores = {
            model.config.id2label[j].lower(): probs[j].item()
            for j in range(len(probs))
        }

        sentiment = (
            scores.get("positive", 0.0)
            - scores.get("negative", 0.0)
        )

        values.append(sentiment)

# Verify results
assert len(values) == len(news)

news["sentiment"] = values

print("Successfully scored", len(news), "articles")
print(news[["headline", "sentiment"]].head())

Device: cpu


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Scoring AAPL headlines:   0%|          | 0/437 [00:00<?, ?it/s]

Successfully scored 3489 articles
                                            headline  sentiment
0               Where Will Apple Stock Be in 1 Year?   0.007390
1     Where Will Unity Software Stock Be in 3 Years?   0.004012
2  4 Red-Hot Growth Stocks to Buy in 2023 and Beyond   0.196620
3  Even in an Advertising Slowdown, These 3 Stock...   0.078827
4  3 Unstoppable Growth Stocks to Buy After a Sto...   0.802019


## 5. Export model-ready data with a conservative availability delay
The price notebook maps to the *strictly next trading day*. Here we additionally delay by 7 calendar days. This is intentionally conservative but **does not repair incorrectly dated articles**. Preserve audit columns to investigate abnormal results. The model input is two columns exactly: `published_at` and `sentiment`. Note: here `published_at` is an *availability proxy*, not the true publication timestamp.

In [7]:
# 5. Export model-ready news data
from pathlib import Path

OUT = Path("/content/AAPL_News_Output")
OUT.mkdir(parents=True, exist_ok=True)
import pandas as pd

DELAY_DAYS = 7

# FIX: Convert publication dates from strings to datetime
news["published_at"] = pd.to_datetime(
    news["published_at"],
    utc=True,
    errors="coerce"
)

# Also convert the original timestamps if present
if "source_timestamp" in news.columns:
    news["source_timestamp"] = pd.to_datetime(
        news["source_timestamp"],
        utc=True,
        errors="coerce"
    )

# Exclude suspicious articles
news["eligible"] = (
    ~news["timestamp_caution"].fillna(False).astype(bool)
    & news["published_at"].notna()
    & news["sentiment"].notna()
)

# Apply seven-day availability delay
news["model_available_at"] = (
    news["published_at"] +
    pd.Timedelta(days=DELAY_DAYS)
)

# Save detailed audit file
news[
    [
        "source_timestamp",
        "published_at",
        "model_available_at",
        "headline",
        "url",
        "publisher",
        "sentiment",
        "date_only_or_midnight",
        "timestamp_caution",
        "eligible"
    ]
].to_csv(
    OUT / "AAPL_news_audit.csv",
    index=False
)

# Create model-ready news dataset
export = (
    news.loc[
        news["eligible"],
        ["model_available_at", "sentiment"]
    ]
    .rename(
        columns={"model_available_at": "published_at"}
    )
    .sort_values("published_at")
)

export.to_csv(
    OUT / "AAPL_news.csv",
    index=False
)

print("Exported", len(export), "Apple news sentiment scores")
print("Saved:", OUT / "AAPL_news.csv")
print("Audit:", OUT / "AAPL_news_audit.csv")
print("Date range:", export["published_at"].min(), "to", export["published_at"].max())

Exported 3400 Apple news sentiment scores
Saved: /content/AAPL_News_Output/AAPL_news.csv
Audit: /content/AAPL_News_Output/AAPL_news_audit.csv
Date range: 2020-03-16 00:00:00+00:00 to 2023-01-07 00:00:00+00:00


## 6. Download outputs and run the stock model
Download `AAPL_news.csv`, `AAPL_news_audit.csv`, and `AAPL_news_year_counts.csv`; upload `AAPL_news.csv` alongside `AAPL Dataset.csv` in the stock forecasting notebook, set `USE_NEWS=True`, then run all. **Compare with `USE_NEWS=False` on identical test dates and record the naive benchmark.** Don't assume an improvement is meaningful until article dates have been validated.

In [8]:
try:
    from google.colab import files
    for name in ['AAPL_news.csv','AAPL_news_audit.csv','AAPL_news_year_counts.csv']:
        files.download(str(OUT/name))
except ImportError:
    print('Files saved to',OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: /content/AAPL_News_Output/AAPL_news_year_counts.csv